In [ ]:
import os, json, glob

dbutils.widgets.text("catalog_name", "main")
dbutils.widgets.text("raw_schema", "iot_raw")
dbutils.widgets.text("bronze_schema", "iot_bronze")
dbutils.widgets.text("config_dir", "")

catalog = dbutils.widgets.get("catalog_name")
raw_schema = dbutils.widgets.get("raw_schema")
bronze_schema = dbutils.widgets.get("bronze_schema")
config_dir = dbutils.widgets.get("config_dir")

def sql_tags_kv(tags: dict) -> str:
    # ('k1'='v1','k2'='v2')
    parts = []
    for k, v in tags.items():
        kk = str(k).replace("'", "''")
        vv = str(v).replace("'", "''")
        parts.append(f"'{kk}' = '{vv}'")
    return ", ".join(parts)

def apply_table_tags(fqtn: str, tags: dict):
    if not tags:
        return
    spark.sql(f"ALTER TABLE {fqtn} SET TAGS ({sql_tags_kv(tags)})")

def apply_column_tags(fqtn: str, col_name: str, tags: dict):
    if not tags:
        return
    col_escaped = col_name.replace("`", "``")
    spark.sql(f"ALTER TABLE {fqtn} ALTER COLUMN `{col_escaped}` SET TAGS ({sql_tags_kv(tags)})")

cfg_paths = sorted(glob.glob(os.path.join(config_dir, "*.json")))
for p in cfg_paths:
    with open(p, "r") as f:
        cfg = json.load(f)

    name = cfg["name"]

    # Base tags from config + override layer per physical table
    base_table_tags = cfg.get("governance", {}).get("table_tags", {})
    raw_tags = dict(base_table_tags)
    raw_tags["layer"] = "raw"

    bronze_tags = dict(base_table_tags)
    bronze_tags["layer"] = "bronze"

    raw_table = f"{catalog}.{raw_schema}.{name}_raw"
    bronze_table = f"{catalog}.{bronze_schema}.{name}"

    # Apply table tags
    apply_table_tags(raw_table, raw_tags)
    apply_table_tags(bronze_table, bronze_tags)

    # Apply column tags (config columns)
    for c in cfg.get("columns", []):
        col_tags = c.get("tags", {})
        apply_column_tags(raw_table, c["name"], col_tags)
        apply_column_tags(bronze_table, c["name"], col_tags)

    # Apply derived column tags only to bronze (if present in cfg)
    for dc in cfg.get("derived_columns", []):
        apply_column_tags(bronze_table, dc["name"], dc.get("tags", {}))

print("Applied UC tags to raw + bronze tables.")